# Popularity of Internet Archive books: EDA and machine learning

**Question.** Can we estimate how much an archived English fiction book is viewed (`views_all_time`) from information about the book, without ever seeing its views?

**What the model sees.** Only information *about* each book: publication year, scanned pages, years on archive.org, number of creators and subjects, three genre flags, the collection flag, and the words in the title and subject tags. It does **not** read the book text, and it never sees any views figure.

**Data.** `data/training_api.csv`: 5,600 books, all English, collected through the Internet Archive API: 200 random benchmark books + 1,800 each of science fiction, fantasy and mystery/thriller.

**Two phases.**
1. *Phase 1:* classify low / medium / high views.
2. *Phase 2:* estimate the number of views (on a log scale; we report how often the estimate is within 2× or 3× of the truth).

**How we test.** A locked test set (about 15% of the books, never used until the end) and 10-fold cross-validation on the rest. All copies, volumes and editions of one work (`work_group`) always stay in the same part. Every number is compared with a simple baseline.

*Running it:* open in Jupyter from the repo (`notebooks/`) or in Google Colab (it asks you to upload `data/training_api.csv`). Set `QUICK = True` in the first code cell for a fast check; the full run takes roughly 10-20 minutes.

*Views measure engagement with one archived copy, not worldwide popularity or sales. The results estimate views of books that already exist; they do not forecast the success of a new book.*

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, f1_score, confusion_matrix, silhouette_score
from IPython.display import display, Markdown

SEED = 2026
QUICK = False                      # True = faster run (5 folds, lighter models, a few minutes) for a quick check
N_FOLDS = 5 if QUICK else 10
ITERS, LR = (80, 0.08) if QUICK else (150, 0.06)       # size of the boosted-tree models
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 140, "display.max_columns", 30)

# --- where is the data? (local repo, or upload the file when running in Google Colab)
CANDIDATES = [Path("../data/training_api.csv"), Path("data/training_api.csv"), Path("training_api.csv")]
if not any(p.exists() for p in CANDIDATES):
    try:
        from google.colab import files            # Colab: choose data/training_api.csv from the repo on your computer
        print("Upload training_api.csv"); files.upload()
    except ImportError:
        raise FileNotFoundError("training_api.csv not found; run from the repo (notebooks/) or place the file next to the notebook")
DATA = next(p for p in CANDIDATES if p.exists())
df = pd.read_csv(DATA, encoding="utf-8-sig")
print(DATA, df.shape)

## 1. Load and validate the data

In [ ]:
assert df["identifier"].is_unique and (df["status"] == "ok").all() and df["language"].str.contains("eng").all()
assert df["views_all_time"].ge(0).all() and df["views_all_time"].notna().all()
print("rows:", len(df), "| distinct work groups:", df["work_group"].nunique(), "| max items per group:", df["work_group"].value_counts().max())
print("rows per sampling stratum:"); display(df["sampling_stratum"].value_counts().to_frame("rows").T)
print("English only (no other language):", (df["language"] == "eng").sum(), "| bilingual with English:", (df["language"] != "eng").sum())
print("missing values in model inputs:"); display(df[["publication_year","scanned_page_count","addeddate","creators","subjects","title"]].isna().sum().to_frame("missing").T)

In [ ]:
# --- features: only information about the book, never any views value
df["added"] = pd.to_datetime(df["addeddate"], utc=True)
df["snapshot"] = pd.to_datetime(df["retrieved_at_utc"], utc=True)
df["age_years"] = (df["snapshot"] - df["added"]).dt.days / 365.25          # years on archive.org when the views were read
df["n_creators"] = df["creators"].fillna("").str.count(r"\|") + 1
df["n_subjects"] = df["subjects"].fillna("").str.count(r"\|") + 1
df["title_words"] = df["title"].fillna("").str.split().str.len()
for c in ["genre_science_fiction", "genre_fantasy", "genre_mystery_thriller", "in_americana"]:
    df[c] = df[c].astype(int)
df["text"] = df["title"].fillna("") + " " + df["subjects"].fillna("").str.replace("|", " ", regex=False)
df["log_views"] = np.log1p(df["views_all_time"])
NUM = ["publication_year", "scanned_page_count", "age_years", "n_creators", "n_subjects", "title_words",
       "genre_science_fiction", "genre_fantasy", "genre_mystery_thriller", "in_americana"]
print("age_years range:", round(df["age_years"].min(), 2), "to", round(df["age_years"].max(), 2))
display(df[NUM].describe().T[["min", "25%", "50%", "75%", "max"]].round(2))

## 2. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
sns.histplot(df["views_all_time"], bins=60, ax=ax[0]); ax[0].set_title("Views (raw): very skewed"); ax[0].set_xlim(0, 3000)
sns.histplot(df["log_views"], bins=50, ax=ax[1]); ax[1].set_title("log(1 + views): much closer to a bell shape")
sns.boxplot(data=df, x="sampling_stratum", y="log_views", ax=ax[2]); ax[2].set_title("log views by sampling group"); ax[2].tick_params(axis="x", rotation=20); ax[2].set_xlabel("")
plt.tight_layout(); plt.show()
display(df.groupby("sampling_stratum")["views_all_time"].describe(percentiles=[.1, .5, .9]).round(0))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
sns.histplot(df["publication_year"], bins=40, ax=ax[0]); ax[0].set_title("Publication year")
sns.histplot(df["scanned_page_count"], bins=40, ax=ax[1]); ax[1].set_title("Scanned pages")
sns.scatterplot(data=df.sample(2000, random_state=SEED), x="age_years", y="log_views", alpha=.35, s=14, ax=ax[2]); ax[2].set_title("Years on archive.org vs log views")
plt.tight_layout(); plt.show()
print("correlation of each number with log views (Spearman):")
display(pd.Series({c: spearmanr(df[c], df["log_views"])[0] for c in NUM}).round(3).sort_values().to_frame("rho").T)

In [ ]:
src = df.assign(source=np.where(df["in_americana"] == 1, "Americana", "other English texts"))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
sns.boxplot(data=src[src["sampling_stratum"] != "benchmark_random_fiction"], x="sampling_stratum", y="log_views", hue="source", ax=ax[0]); ax[0].set_title("Same genre, different source collection"); ax[0].set_xlabel("")
overlap = df.loc[df["sampling_stratum"] != "benchmark_random_fiction", ["genre_science_fiction", "genre_fantasy", "genre_mystery_thriller"]].sum(axis=1).value_counts().sort_index()
overlap.plot.bar(ax=ax[1]); ax[1].set_title("Genre rules matched per book (genre books)"); ax[1].set_xlabel("number of genres matched")
plt.tight_layout(); plt.show()
print("median views, Americana vs other:", src.groupby("source")["views_all_time"].median().to_dict())
print("year trend: median views by publication decade"); display(df.groupby((df["publication_year"] // 20) * 20)["views_all_time"].agg(["count", "median"]).T.round(0))

**What the EDA tells us.** Views are extremely skewed (a few books have tens of thousands, most have under a few hundred), so we model `log(1 + views)`. The books added to archive.org longest ago have the most views, which is a large part of what any model will learn. The Americana books and the other English texts have different view levels, which is why the collection flag is kept and tested.

## 3. Task definitions and split

* **Phase 1 labels:** low / medium / high are the bottom third, middle third and top third of views. The two cut-offs are computed **from the training part only** and then applied unchanged to the unseen books (this is repeated inside every cross-validation fold).
* **Phase 2 target:** `log(1 + views)`.
* **Split:** about 15% of the books are locked away as the final test set (grouped by `work_group`, balanced by sampling group). The other 85% is the training pool, used with 10-fold cross-validation to compare models and tune choices.

In [ ]:
outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)       # one of 7 parts is about 14% of the books
pool_idx, test_idx = next(outer.split(df, df["sampling_stratum"], df["work_group"]))
pool, test = df.iloc[pool_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)
assert set(pool["work_group"]).isdisjoint(set(test["work_group"]))
print(f"training pool: {len(pool)} books | locked test set: {len(test)} books ({len(test)/len(df):.1%}) | shared work groups: 0")
display(pd.concat({"pool": pool["sampling_stratum"].value_counts(), "test": test["sampling_stratum"].value_counts()}, axis=1))

In [ ]:
def preprocess(cols, text=True, scale=False):
    num_cols = [c for c in cols if c != "text"]          # "text" is handled by TF-IDF, never as a number
    parts = [("num", StandardScaler() if scale else "passthrough", num_cols)]
    if text:
        parts.append(("txt", TfidfVectorizer(max_features=200, min_df=5, token_pattern=r"[A-Za-z]{3,}", stop_words="english"), "text"))
    return ColumnTransformer(parts, sparse_threshold=0)

def reg_models(cols):
    return {
        "baseline (always the typical value)": lambda: Pipeline([("p", preprocess(cols, False)), ("m", DummyRegressor(strategy="median"))]),
        "Ridge (numbers + words)": lambda: Pipeline([("p", preprocess(cols, True, True)), ("m", Ridge(alpha=30))]),
        "Boosted trees (numbers only)": lambda: Pipeline([("p", preprocess(cols, False)), ("m", HistGradientBoostingRegressor(max_iter=ITERS, learning_rate=LR, random_state=SEED))]),
        "Boosted trees (numbers + words)": lambda: Pipeline([("p", preprocess(cols, True)), ("m", HistGradientBoostingRegressor(max_iter=ITERS, learning_rate=LR, random_state=SEED))]),
    }
def clf_models(cols):
    return {
        "baseline (always the commonest class)": lambda: Pipeline([("p", preprocess(cols, False)), ("m", DummyClassifier(strategy="most_frequent"))]),
        "Logistic regression (numbers + words)": lambda: Pipeline([("p", preprocess(cols, True, True)), ("m", LogisticRegression(max_iter=3000, C=0.3))]),
        "Boosted trees (numbers only)": lambda: Pipeline([("p", preprocess(cols, False)), ("m", HistGradientBoostingClassifier(max_iter=ITERS, learning_rate=LR, random_state=SEED))]),
        "Boosted trees (numbers + words)": lambda: Pipeline([("p", preprocess(cols, True)), ("m", HistGradientBoostingClassifier(max_iter=ITERS, learning_rate=LR, random_state=SEED))]),
    }
FEATS = NUM + ["text"]

def cutoffs(views):                      # low / medium / high from the training views only
    return np.quantile(views, [1/3, 2/3])
def to_class(views, cut):
    return np.where(views <= cut[0], 0, np.where(views >= cut[1], 2, 1))

def folds(data, n=None, seed=SEED):
    n = n or N_FOLDS
    return list(StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed).split(data, data["sampling_stratum"], data["work_group"]))

def cv_regression(make, cols, data, n=None):
    oof = np.zeros(len(data)); y = data["log_views"].values
    for tr, te in folds(data, n):
        m = make(); m.fit(data.iloc[tr][cols], y[tr]); oof[te] = m.predict(data.iloc[te][cols])
    return oof

def cv_classification(make, cols, data, n=None):
    oof = np.zeros(len(data), int); truth = np.zeros(len(data), int); v = data["views_all_time"].values
    for tr, te in folds(data, n):
        cut = cutoffs(v[tr]); m = make(); m.fit(data.iloc[tr][cols], to_class(v[tr], cut))
        oof[te] = m.predict(data.iloc[te][cols]); truth[te] = to_class(v[te], cut)
    return oof, truth

def reg_metrics(y, p):
    err = np.abs(p - y)
    return {"R2": r2_score(y, p), "MAE (log)": mean_absolute_error(y, p), "typical error (x times)": float(np.exp(np.median(err))),
            "within 2x": float((err <= np.log(2)).mean()), "within 3x": float((err <= np.log(3)).mean()),
            "Spearman": spearmanr(y, p)[0] if np.std(p) > 0 else 0.0}
def clf_metrics(t, p):
    return {"accuracy": accuracy_score(t, p), "macro-F1": f1_score(t, p, average="macro"), "not off by two classes": float((np.abs(p - t) < 2).mean())}

## 4. Phase 1: low / medium / high (10-fold cross-validation on the training pool)

In [ ]:
rows, store1 = [], {}
for name, make in clf_models(FEATS).items():
    oof, truth = cv_classification(make, FEATS, pool)
    rows.append({"model": name, **clf_metrics(truth, oof)}); store1[name] = (oof, truth)
phase1 = pd.DataFrame(rows).set_index("model")
display(phase1.round(3))
best1 = phase1.drop(index=[i for i in phase1.index if i.startswith("baseline")])["macro-F1"].idxmax()
print("best by cross-validated macro-F1:", best1)
oof, truth = store1[best1]
cm = confusion_matrix(truth, oof, normalize="true")
plt.figure(figsize=(4.4, 3.6)); sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=["low", "medium", "high"], yticklabels=["low", "medium", "high"])
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Phase 1 confusion matrix (share of each true class)"); plt.show()

## 5. Phase 2: how many views? (10-fold cross-validation on the training pool)

In [ ]:
rows, store2 = [], {}
y_pool = pool["log_views"].values
for name, make in reg_models(FEATS).items():
    oof = cv_regression(make, FEATS, pool)
    rows.append({"model": name, **reg_metrics(y_pool, oof)}); store2[name] = oof
phase2 = pd.DataFrame(rows).set_index("model")
display(phase2.round(3))
best2 = phase2.drop(index=[i for i in phase2.index if i.startswith("baseline")])["R2"].idxmax()
print("best by cross-validated R2:", best2)
p = store2[best2]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(y_pool, p, s=6, alpha=.3); lim = [y_pool.min(), y_pool.max()]; ax[0].plot(lim, lim, "r--", lw=1)
ax[0].set_xlabel("true log(1 + views)"); ax[0].set_ylabel("predicted"); ax[0].set_title("Predicted vs true (out-of-fold)")
sns.histplot(np.expm1(p) / np.expm1(y_pool), bins=np.logspace(-1.5, 1.5, 50), ax=ax[1], log_scale=True)
ax[1].axvline(1, color="r", ls="--"); ax[1].set_title("Predicted views / true views"); plt.tight_layout(); plt.show()

## 6. Which information matters? (ablation and permutation importance)

In [ ]:
make_full = reg_models(FEATS)["Boosted trees (numbers + words)"]
variants = {
    "all information": FEATS,
    "without years on archive.org": [c for c in FEATS if c != "age_years"],
    "without title/subject words": [c for c in FEATS if c != "text"],
    "without genre flags": [c for c in FEATS if not c.startswith("genre_")],
    "without collection flag": [c for c in FEATS if c != "in_americana"],
    "only year + pages + years on archive.org": ["publication_year", "scanned_page_count", "age_years"],
}
rows = []
for label, cols in variants.items():
    num_only = [c for c in cols if c != "text"]
    make = (lambda cols=cols, num_only=num_only: Pipeline([("p", preprocess(num_only, "text" in cols)), ("m", HistGradientBoostingRegressor(max_iter=ITERS, learning_rate=LR, random_state=SEED))]))
    rows.append({"inputs": label, **{k: v for k, v in reg_metrics(y_pool, cv_regression(make, cols, pool)).items() if k in ("R2", "MAE (log)", "within 2x", "Spearman")}})
ablation = pd.DataFrame(rows).set_index("inputs"); display(ablation.round(3))

## 7. Final test on the locked books (used once)

In [ ]:
final_make = reg_models(FEATS)[best2]
reg_final = final_make(); reg_final.fit(pool[FEATS], pool["log_views"])
pt = reg_final.predict(test[FEATS]); yt = test["log_views"].values
base_reg = reg_models(FEATS)["baseline (always the typical value)"]().fit(pool[FEATS], pool["log_views"]).predict(test[FEATS])

cut = cutoffs(pool["views_all_time"].values)
clf_final = clf_models(FEATS)[best1](); clf_final.fit(pool[FEATS], to_class(pool["views_all_time"].values, cut))
ct = to_class(test["views_all_time"].values, cut); cp = clf_final.predict(test[FEATS])
base_clf = clf_models(FEATS)["baseline (always the commonest class)"]().fit(pool[FEATS], to_class(pool["views_all_time"].values, cut)).predict(test[FEATS])
print(f"views cut-offs from the training pool: low <= {cut[0]:.0f}, high >= {cut[1]:.0f}")
final = pd.DataFrame({
    "phase 1 (class)": {"baseline accuracy": accuracy_score(ct, base_clf), "model accuracy": accuracy_score(ct, cp), "baseline macro-F1": f1_score(ct, base_clf, average="macro"), "model macro-F1": f1_score(ct, cp, average="macro")},
}).T
display(final.round(3))
display(pd.DataFrame({"baseline": reg_metrics(yt, base_reg), "model": reg_metrics(yt, pt)}).round(3))
cv_ref = phase2.loc[best2]
print(f"cross-validation said R2 = {cv_ref['R2']:.3f}, within 2x = {cv_ref['within 2x']:.3f}; locked test says R2 = {r2_score(yt, pt):.3f}, within 2x = {(np.abs(pt-yt) <= np.log(2)).mean():.3f}")

# results per genre on the locked test books
by = []
for g, sub in test.groupby("sampling_stratum"):
    ix = sub.index.values
    by.append({"group": g, "books": len(ix), "phase 1 accuracy": accuracy_score(ct[ix], cp[ix]), "phase 2 R2": r2_score(yt[ix], pt[ix]) if len(ix) > 5 else np.nan, "within 2x": (np.abs(pt[ix]-yt[ix]) <= np.log(2)).mean()})
display(pd.DataFrame(by).set_index("group").round(3))

In [ ]:
imp = permutation_importance(reg_final, test[FEATS], yt, n_repeats=5, random_state=SEED, scoring="r2")
imp_df = pd.Series(imp.importances_mean, index=FEATS).sort_values()
plt.figure(figsize=(7, 4)); imp_df.plot.barh(); plt.xlabel("drop in R2 when this input is shuffled (locked test set)"); plt.title("Permutation importance"); plt.tight_layout(); plt.show()

## 8. Would more books help? (learning curve)

In [ ]:
rng = np.random.RandomState(SEED)
fracs = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
rows = []
make_r = reg_models(FEATS)["Boosted trees (numbers + words)"]; make_c = clf_models(FEATS)["Boosted trees (numbers + words)"]
for tr, te in folds(pool, 5):
    cut_f = cutoffs(pool["views_all_time"].values[tr])
    for f in fracs:
        sub = rng.choice(tr, int(len(tr) * f), replace=False)
        m = make_r(); m.fit(pool.iloc[sub][FEATS], pool["log_views"].values[sub]); r2 = r2_score(pool["log_views"].values[te], m.predict(pool.iloc[te][FEATS]))
        c = make_c(); c.fit(pool.iloc[sub][FEATS], to_class(pool["views_all_time"].values[sub], cut_f)); acc = accuracy_score(to_class(pool["views_all_time"].values[te], cut_f), c.predict(pool.iloc[te][FEATS]))
        rows.append({"share of training pool": f, "training books": len(sub), "R2": r2, "accuracy": acc})
lc_plot = pd.DataFrame(rows).groupby("share of training pool").mean().set_index("training books", drop=True)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
lc_plot["R2"].plot(marker="o", ax=ax[0]); ax[0].set_title("Phase 2: R2 vs training size"); ax[0].set_xlabel("training books")
lc_plot["accuracy"].plot(marker="o", ax=ax[1]); ax[1].set_title("Phase 1: accuracy vs training size"); ax[1].set_xlabel("training books")
plt.tight_layout(); plt.show()
display(lc_plot.round(3).T)

## 9. Optional extra: K-Means (exploratory, not the predictor)

In [ ]:
km_x = pd.DataFrame({"year": df["publication_year"], "log pages": np.log(df["scanned_page_count"])})
z = StandardScaler().fit_transform(km_x)
sil = {k: silhouette_score(z, KMeans(k, n_init=10, random_state=SEED).fit_predict(z), sample_size=3000, random_state=SEED) for k in range(2, 8)}
print("silhouette by k:", {k: round(v, 3) for k, v in sil.items()})
k = max(sil, key=sil.get); df["cluster"] = KMeans(k, n_init=10, random_state=SEED).fit_predict(z)
prof = df.groupby("cluster").agg(books=("identifier", "size"), median_year=("publication_year", "median"), median_pages=("scanned_page_count", "median"),
                                 median_views=("views_all_time", "median"), share_sf=("genre_science_fiction", "mean"), share_fantasy=("genre_fantasy", "mean"), share_mystery=("genre_mystery_thriller", "mean")).round(2)
print("chosen k =", k, "(views and genre were NOT used for clustering; they only describe the clusters)"); display(prof)
plt.figure(figsize=(6, 4)); sns.scatterplot(data=df.sample(2500, random_state=SEED), x="publication_year", y="scanned_page_count", hue="cluster", palette="tab10", s=16); plt.yscale("log"); plt.title("K-Means clusters"); plt.show()

## 10. Summary and limits

In [ ]:
a1, m1 = phase1.loc[[i for i in phase1.index if i.startswith("baseline")][0], "accuracy"], phase1.loc[best1, "accuracy"]
display(Markdown(f'''
**Results (training pool cross-validation; locked-test values are in section 7).**

* **Phase 1:** best model *{best1}*: accuracy {m1:.1%} versus {a1:.1%} for the baseline (about one in three is what guessing gives); macro-F1 {phase1.loc[best1, "macro-F1"]:.3f}.
* **Phase 2:** best model *{best2}*: R² {phase2.loc[best2, "R2"]:.2f} (baseline {phase2.loc[[i for i in phase2.index if i.startswith("baseline")][0], "R2"]:.2f}); the estimate is within 2× of the true views for {phase2.loc[best2, "within 2x"]:.0%} of the books and within 3× for {phase2.loc[best2, "within 3x"]:.0%}; typical error about {phase2.loc[best2, "typical error (x times)"]:.1f}×.
* **What matters most:** see the ablation table and the permutation importance plot (years on archive.org is the strongest single input).

**Limits.**
1. The model only sees information about each book, not its text; popularity also depends on things we cannot see (school reading lists, famous authors, links from other sites).
2. A large part of the signal is how long a book has been on archive.org; the model estimates views of *existing* books, it does not forecast how a *new* book will do.
3. Views are engagement with one archived copy, not sales or worldwide popularity.
4. The books outside the Americana collection have lower views; the collection flag is in the model and tested in the ablation, but a different mix of sources could shift results.
5. Genre labels come from subject-tag rules and are imperfect; 359 books match more than one genre.
6. Descriptions were not used (only about 60% of a sample of books had a real description of 25+ words); they are a possible next step.
'''))